# jevpandas · notebook test

Ask Jev/TypeSafe natural-language questions against a pandas dataframe, from a notebook.

The dataset is `data/incidents.parquet` — 1,000 synthetic incidents (a plain CSV with the same
rows is provided too). Three paths:
- **Live**: calls the real endpoint with your API key.
- **Local**: your own OpenJev/SemIf server, no hosted key.
- **Offline**: a mocked endpoint, no server, no key — proves the wiring works end-to-end.

In [ ]:
# Installs everything (including pandas and tqdm) into the project's .venv.
# Launch Jupyter with `uv run jupyter lab` so the kernel IS that .venv.
!uv sync --extra dev --extra notebook

In [ ]:
import pandas as pd

# Load the synthetic incident set (1,000 rows). Both formats carry the same data:
df = pd.read_parquet(
    "../data/incidents.parquet"
)  # compact, fast (needs pyarrow, in the notebook extra)
# df = pd.read_csv("data/incidents.csv")        # plain CSV, same rows

df.shape  # (1000, 7): incident_id, created_at, service, region, version, subject, message

## Live test (official Jev endpoint)

Set your key before running. Defaults: `https://api.typesafe.ai/v1`, model `jev-latest`.

In [ ]:
from jevpandas import JevClient, JevFrame, choice, noul, score, tqdm_progress

client = JevClient(api_key="YOUR_KEY")  # defaults to https://api.typesafe.ai/v1, jev-latest

# A sample keeps the demo quick; drop .sample(...) to run the full 1,000 rows.
frame = JevFrame(df.sample(20, random_state=1), client)
result = frame.ask(
    {
        "access": noul("Is this a current, unresolved customer access problem?"),
        "topic": choice(
            "Main topic?", {"access": "Login/auth", "billing": "Payments", "other": "Other"}
        ),
        "severity": score("Current disruption?", ["None", "Impaired", "Outage"]),
    },
    columns=["subject", "message"],
    workers=4,
    progress=tqdm_progress(),
)
result  # JevResult: summary line + enriched table

In [ ]:
result[["subject", "access_probability", "topic_label", "severity_value"]]

In [ ]:
result.metadata  # rows, elapsed_seconds, cache_hits, errors, model

## Local OpenJev / SemIf server

Point at your own deployment — no hosted API key needed. Replace `<host>` with the server address.

In [ ]:
from jevpandas import JevClient, JevFrame

client = JevClient(
    base_url="http://<host>:8000",  # your local OpenJev/SemIf server
    api_key="dummy",
    model="openjev-0.1",
)

# Local calls are free, so evaluate a bigger slice; drop .sample(...) for all 1,000 rows.
result = JevFrame(df.sample(100, random_state=1), client).evaluate(
    "Is this a current, unresolved customer access problem?",
    columns=["subject", "message"],
    workers=4,
)
result

## Offline test (no server, no API key)

A mocked endpoint proves the pipeline works before spending any calls.

In [ ]:
import json

import httpx
import pandas as pd

from jevpandas import JevClient, JevFrame


def fake(request):
    state = json.loads(request.content)["state"]
    prob = 0.9 if "login" in state["message"].lower() else 0.05
    name = next(iter(json.loads(request.content)["questions"]))
    return httpx.Response(
        200,
        json={
            "model": "fake-1",
            "answers": {name: {"type": "noul", "noul": prob}},
        },
    )


client = JevClient(transport=httpx.MockTransport(fake))
df = pd.DataFrame({"message": ["cannot login", "refund please"]})
result = JevFrame(df, client).evaluate("Access problem?", columns=["message"])
result  # probabilities 0.9 and 0.05

## Notes

- The fixture is `data/incidents.parquet` (1,000 synthetic rows); `data/incidents.csv` holds the same rows if you prefer plain text.
- Reuse one `JevClient` across cells to benefit from the in-memory cache (`result.metadata["cache_hits"]` shows reuse).
- Point at a local OpenJev/SemIf server with `JevClient(base_url="http://<host>:8000", api_key="dummy", model="openjev-0.1")`.
- `result` is a `pd.DataFrame` subclass, so `.groupby`, `.to_csv`, etc. work as usual.